# 00 — Генерация данных
Наполняет источники синтетическими финтех-данными:- батч (клиенты, счета) → CSV в `landing/`- поток (транзакции) → Kafka topic `transactions`Запустите этот ноутбук **первым**, до `01_ingestion`.

## 1. Зависимости генератора
Нужны только генератору; ставятся в контейнер разово.

In [1]:
!pip install -q faker kafka-python

In [2]:
!pip install kafka-python

## 2. Запуск генератора
Объёмы заданы в `data-generator/generate.py` (по умолчанию 10k клиентов, 200k транзакций).

In [3]:
!python /home/jovyan/data-generator/generate.py

=== Генерация финтех-данных ===
Батч записан (с браком и дублями):
  clients : /home/jovyan/work/landing/clients.csv (10300 строк, включая дубли)
  accounts: /home/jovyan/work/landing/accounts.csv (15419 строк, включая дубли)
Отправка ~200000 транзакций (с браком и дублями) в Kafka topic 'transactions' (kafka:19092)...
  отправлено 20000
  отправлено 40000
  отправлено 60000
  отправлено 80000
  отправлено 100000
  отправлено 120000
  отправлено 140000
  отправлено 160000
  отправлено 180000
  отправлено 200000
Готово: 205825 сообщений отправлено в Kafka (включая дубли).
=== Завершено за 5.6 сек ===


## 3. Проверка: батч приземлился в landing

In [4]:
import os
landing = "/home/jovyan/work/landing"
for f in sorted(os.listdir(landing)):
    size = os.path.getsize(os.path.join(landing, f))
    print(f"{f:15s} {size:>10} байт")

print()
print("=== первые строки clients.csv ===")
with open(os.path.join(landing, "clients.csv"), encoding="utf-8") as fh:
    for _ in range(3):
        print(fh.readline().rstrip())

accounts.csv        704259 байт
clients.csv         716222 байт

=== первые строки clients.csv ===
client_id,full_name,city,segment,reg_date
3673,Наталья Кузнецов,Ростов-на-Дону,mass,2024-11-29
9289,Анна Попов,Новосибирск,mass,2023-09-25


## ГотовоБатч в `landing/`, поток в Kafka. 
Дальше — ноутбук `01_ingestion`:он прочитает оба источника и сложит сырьё в bronze.